In [12]:
%%writefile lab8_llm.py
"""lab8_llm.py - the one place where Lab 8 talks to a language model.

Pick a backend with the environment variable LAB_LLM:
    mock    (default) an offline stand-in written in Python. It is NOT a language model.
    ollama  a model running on your own computer (https://ollama.com)
    groq    a hosted model; needs a free key in the environment variable GROQ_API_KEY

Every call is appended to LOG, so you can always show what was sent and received.
Never write an API key into this file or into a notebook you submit.
"""
import json
import os
import random
import re
import time
import urllib.request

BACKEND = os.environ.get("LAB_LLM", "mock").lower()
MODEL = os.environ.get("LAB_MODEL", "")
LOG = []                      # one dictionary per call: prompt, reply, seconds
_rng = random.Random(0)       # only the mock uses randomness
_mode = "normal"              # the mock can also be switched to a "stuck" mode (Task 4)


def set_seed(seed, mode="normal"):
    """Make the mock repeatable. A real model ignores this."""
    global _rng, _mode
    _rng = random.Random(seed)
    _mode = mode


def ask(prompt, system="", temperature=0.0):
    """Send one prompt, return the reply text, and record the call in LOG."""
    start = time.perf_counter()
    if BACKEND == "mock":
        reply = _mock(prompt)
    elif BACKEND == "ollama":
        reply = _ollama(prompt, system, temperature)
    elif BACKEND == "groq":
        reply = _groq(prompt, system, temperature)
    else:
        raise ValueError("LAB_LLM must be mock, ollama or groq, not " + BACKEND)
    LOG.append({"prompt": prompt, "reply": reply,
                "seconds": round(time.perf_counter() - start, 3)})
    return reply


# ---------------------------------------------------------------- real backends
def _post(url, payload, headers):
    headers = dict(headers, **{"Content-Type": "application/json",
                               "User-Agent": "cgd460-lab8"})
    req = urllib.request.Request(url, json.dumps(payload).encode(), headers)
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode())


def _messages(prompt, system):
    msgs = [{"role": "system", "content": system}] if system else []
    return msgs + [{"role": "user", "content": prompt}]


def _ollama(prompt, system, temperature):
    base = os.environ.get("OLLAMA_URL", "http://localhost:11434")
    out = _post(base + "/api/chat",
                {"model": MODEL or "llama3.2:3b", "stream": False,
                 "messages": _messages(prompt, system),
                 "options": {"temperature": temperature}}, {})
    return out["message"]["content"]


def _groq(prompt, system, temperature):
    base = os.environ.get("GROQ_URL", "https://api.groq.com/openai/v1")
    key = os.environ["GROQ_API_KEY"]          # a KeyError here means: set your key
    out = _post(base + "/chat/completions",
                {"model": MODEL or "llama-3.1-8b-instant", "temperature": temperature,
                 "messages": _messages(prompt, system)},
                {"Authorization": "Bearer " + key})
    return out["choices"][0]["message"]["content"]


# ------------------------------------------------------------------- the mock
CATEGORY_ACTIONS = {
    "Physics / Collision": ("High", "Check collider and collision layer"),
    "Performance": ("Medium", "Profile the scene and check asset sizes"),
    "UI": ("Low", "Check the canvas and event wiring"),
    "Game Logic": ("High", "Trace the state change in the script"),
    "Audio": ("Low", "Check the audio source and play calls"),
}
KEYWORDS = [("Audio", ["sound", "music", "footstep"]),
            ("UI", ["button", "health bar", "screen"]),
            ("Performance", ["freez", "fps", "frame rate", "texture", "loading"]),
            ("Physics / Collision", ["falls through", "walks through", "collid"])]
PLANS = {
    "enemy ai": [("Define behaviour", "Patrol, chase and retreat are written down"),
                 ("Choose states", "Each behaviour is a named state"),
                 ("Implement logic", "Events switch the enemy between states"),
                 ("Test scenarios", "Player distance and obstacles are tested")],
    "playable demo": [("List required features", "Every must-have feature is on the list"),
                      ("Cut optional work", "Nothing optional is left in the plan"),
                      ("Build and integrate", "The features work together in one build"),
                      ("Test on a clean computer", "The demo starts without the editor"),
                      ("Fix blocking bugs", "No crash or dead end remains")],
    "save system": [("Choose what to save", "A list of saved values exists"),
                    ("Write the save file", "A file appears after saving"),
                    ("Load the save file", "The loaded game matches the saved one")],
}
BAD_CALLS = {"Physics.Raycast": "Physics.RaycastSmart(origin, direction);",
             "transform.Translate": "transform.MoveTowardsSmooth(target);",
             "Destroy": "Destroy.After(gameObject, 2f);",
             "audioSource.Play": "audioSource.PlayLoudly();",
             "transform.Find": "transform.FindDeepChild(\"Weapon\");"}
GOOD_LINES = {"Physics.Raycast": "Physics.Raycast(origin, direction, out hit);",
              "transform.Translate": "transform.Translate(move * speed * Time.deltaTime);",
              "Destroy": "Destroy(gameObject, 2f);",
              "audioSource.Play": "audioSource.Play();",
              "transform.Find": "transform.Find(\"Weapon\");"}
PHRASES = [("cast a ray", "Physics.Raycast"), ("move the character", "transform.Translate"),
           ("destroy", "Destroy"), ("play the sound", "audioSource.Play"),
           ("find the child", "transform.Find")]
LOG_TEXT = ("ERROR NullReferenceException: Object reference not set to an instance of an object\n"
            "  at InventoryUI.OpenInventory() in InventoryUI.cs:line 42")


def _classify(report):
    text = report.lower()
    for category, words in KEYWORDS:
        if any(w in text for w in words):
            return category
    return "Game Logic"


def _mock(prompt):
    r = _rng
    # --- bug triage: few-shot prompts end with "Output:", zero-shot prompts do not
    if "Report:" in prompt or prompt.rstrip().endswith("Output:"):
        report = prompt.rstrip().split("Report:" if "Report:" in prompt else "Input:")[-1]
        report = report.replace("Output:", "").strip()
        few = prompt.rstrip().endswith("Output:")
        cat = _classify(report)
        if r.random() < (0.05 if few else 0.15):
            cat = r.choice([c for c in CATEGORY_ACTIONS if c != cat])
        sev, act = CATEGORY_ACTIONS[cat]
        if r.random() < (0.95 if few else 0.40):
            return f"{cat} | {sev} | {act}"
        return (f"This looks like a {cat} problem. I would rate the severity as {sev}, "
                f"and a good first step is to {act[0].lower() + act[1:]}.")
    # --- ReAct
    if "Available tools" in prompt:
        seen = prompt.split("Question:")[-1]
        if _mode == "stuck":
            return "Thought: I should read the log again.\nAction: read_log[]"
        if r.random() < 0.10:
            return "Thought: I will run the game and look.\nAction: run_game[]"
        if "slot.item.name" in seen:
            return ("Thought: Line 42 reads slot.item.name, so slot.item is null for an empty slot.\n"
                    "Final: In InventoryUI.cs line 42, check that slot.item is not null before reading name.")
        if "NullReferenceException" in seen:
            return ("Thought: The error points to InventoryUI.cs line 42. I should read that file.\n"
                    "Action: search_file[InventoryUI.cs]")
        return "Thought: I need the error message first.\nAction: read_log[]"
    # --- context experiment
    if "Question:" in prompt and "crash" in prompt.lower():
        if "NullReferenceException" not in prompt:
            return ("I cannot be sure without more information. Common causes of a crash "
                    "when a menu opens are null references, missing assets or endless loops.")
        reply = "Likely cause: a NullReferenceException at InventoryUI.cs:42, so something on that line is empty."
        if "slot.item.name" in prompt:
            reply += " The line reads slot.item.name, so slot.item is probably null for an empty slot. Check it for null first."
        if "Unity 2022.3" in prompt:
            reply += " Noted: Unity 2022.3 LTS."
        return reply
    # --- planning
    if "Goal:" in prompt and "JSON" in prompt:
        goal = prompt.split("Goal:")[1].splitlines()[0].lower()
        steps = next(v for k, v in PLANS.items() if k in goal)
        retry = "only" in prompt and "last answer" in prompt
        x = r.random()
        if x < (0.03 if retry else 0.20):
            return "\n".join(f"{i}. {s}: {c}" for i, (s, c) in enumerate(steps, 1))
        items = [{"step": s, "check": c} for s, c in steps]
        if x < (0.06 if retry else 0.30):
            del items[r.randrange(len(items))]["check"]
        body = json.dumps(items, indent=1)
        return "```json\n" + body + "\n```" if r.random() < 0.15 else body
    # --- code generation (hallucination check)
    if "Write one line of C#" in prompt:
        key = next(k for p, k in PHRASES if p in prompt.lower())
        x = r.random()
        if x < 0.25:
            return BAD_CALLS[key]
        if x < 0.35 and key == "Destroy":
            return "Destroy(2f, gameObject);"       # a real function used wrongly
        return GOOD_LINES[key]
    if "hello" in prompt.lower():
        return "Hello from the Lab 8 stand-in. I am a Python function, not a language model."
    return "(mock) I do not have a scripted answer for this prompt."

Overwriting lab8_llm.py


In [13]:
"""lab8_work.py - CGD-460L Lab 8: LLMs as Reasoning Agents.

Contains Activities 1-3 and Tasks 1-5, plus the figures needed for the
Task 6 worksheet. Backend is chosen with LAB_LLM (default: mock).
No API key is stored in this file; keys are read from the environment only.
"""
import json
import re
import statistics
import lab8_llm as llm
from lab8_llm import ask


print("Backend used:", llm.BACKEND, "| model:", llm.MODEL or "(default)")

Backend used: mock | model: (default)


In [14]:
llm.set_seed(0)
print(ask("Say hello to the player in one short sentence."))
call = llm.LOG[-1]
print("calls so far:", len(llm.LOG), "| seconds:", call["seconds"],
      "| reply length:", len(call["reply"]))

Hello from the Lab 8 stand-in. I am a Python function, not a language model.
calls so far: 262 | seconds: 0.0 | reply length: 76


In [15]:
QUESTION = "Why does the game crash when the player opens the inventory?"
CONTEXT = {
    "log": "Error log: " + llm.LOG_TEXT.replace("\n", " "),
    "script": "Script line: 42: label.text = slot.item.name;",
    "version": "Engine version: Unity 2022.3 LTS",
}


def build_prompt(question, items):
    parts = [CONTEXT[name] for name in items]
    return "\n".join(parts + ["Question: " + question])


for items in [[], ["log"]]:
    prompt = build_prompt(QUESTION, items)
    reply = ask(prompt)
    print(f"{items or 'none'}: {len(prompt)} chars")
    print("  reply:", reply)

none: 70 chars
  reply: I cannot be sure without more information. Common causes of a crash when a menu opens are null references, missing assets or endless loops.
['log']: 223 chars
  reply: Likely cause: a NullReferenceException at InventoryUI.cs:42, so something on that line is empty.


In [16]:


def clean_json(text):
    """Remove a ```json fence if the model added one."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    return text.strip()


def check_plan(text):
    """Return (plan, problem). plan is None when the reply is not usable."""
    try:
        plan = json.loads(clean_json(text))
    except json.JSONDecodeError:
        return None, "not valid JSON"
    if not isinstance(plan, list) or not 3 <= len(plan) <= 6:
        return None, "needs 3 to 6 steps"
    if not all(isinstance(s, dict) and {"step", "check"} <= s.keys() for s in plan):
        return None, "a step is missing 'step' or 'check'"
    return plan, ""


def plan_prompt(goal):
    return (f"Goal: {goal}\nSplit the goal into 3 to 6 steps. Reply with a JSON array only. "
            'Each item must look like {"step": "...", "check": "how to know it is done"}.')


for seed in (4, 5):
    llm.set_seed(seed)
    plan, problem = check_plan(ask(plan_prompt("Create a simple enemy AI prototype")))
    print(f"seed {seed}: problem = {problem or 'none'}")
    for i, item in enumerate(plan or [], 1):
        print(f" {i}. {item['step']} -> check: {item['check']}")

seed 4: problem = a step is missing 'step' or 'check'
seed 5: problem = none
 1. Define behaviour -> check: Patrol, chase and retreat are written down
 2. Choose states -> check: Each behaviour is a named state
 3. Implement logic -> check: Events switch the enemy between states
 4. Test scenarios -> check: Player distance and obstacles are tested


In [17]:
REPORTS = [
    ("Player falls through the floor after a jump", "Physics / Collision"),
    ("Enemy walks through a wall", "Physics / Collision"),
    ("Game freezes for five seconds when loading level 3", "Performance"),
    ("Frame rate drops to 12 fps in the forest scene", "Performance"),
    ("Textures take a long time to appear", "Performance"),
    ("Boss health bar is not shown on the screen", "UI"),
    ("Inventory button does not respond when clicked", "UI"),
    ("Score resets to zero after pausing", "Game Logic"),
    ("Enemy keeps patrolling after the player is dead", "Game Logic"),
    ("Door stays closed even after the key is collected", "Game Logic"),
    ("Footstep sound plays twice for each step", "Audio"),
    ("Music stops when the menu is opened", "Audio"),
]
CATEGORIES = "Physics / Collision, Performance, UI, Game Logic, Audio"
ZERO = ("Classify this bug report as one of: " + CATEGORIES + ". "
        "Also give a severity (Low, Medium or High) and one action.\nReport: {report}")
FEW = ("Classify bug reports. Use exactly the format: Category | Severity | Action\n"
       "Input: Player falls through the floor\n"
       "Output: Physics / Collision | High | Check collider and collision layer\n"
       "Input: Loading screen freezes the game\n"
       "Output: Performance | Medium | Profile the scene and check asset sizes\n"
       "Input: {report}\nOutput:")
PATTERN = re.compile(r"^(Physics / Collision|Performance|UI|Game Logic|Audio) \| (Low|Medium|High) \| (.+)$")


def parse(reply):
    m = PATTERN.match(reply.strip())
    return (m.group(1), m.group(2), m.group(3)) if m else None


def run_method(template, seed):
    """One full pass over the 12 reports. Returns (valid_count, correct_count)."""
    llm.set_seed(seed)
    valid = correct = 0
    for report, truth in REPORTS:
        parsed = parse(ask(template.format(report=report)))
        if parsed:
            valid += 1
            correct += parsed[0] == truth
    return valid, correct


task1 = {}
for name, template in [("zero-shot", ZERO), ("few-shot", FEW)]:
    results = [run_method(template, seed) for seed in range(10, 20)]
    valid = sum(v for v, _ in results)
    correct = sum(c for _, c in results)
    total = 10 * len(REPORTS)
    task1[name] = (valid, correct, total)
    print(f"{name}: format valid {valid}/{total} ({valid / total:.0%}), "
          f"correct among valid {correct}/{valid} ({correct / valid:.0%}), "
          f"correct overall {correct}/{total} ({correct / total:.0%})")

# Extra evidence for the write-up: print five raw zero-shot replies (seed 10)
print("\nFive raw zero-shot replies (seed 10) - what would the parser do?")
llm.set_seed(10)
for report, _ in REPORTS[:5]:
    raw = ask(ZERO.format(report=report))
    print(f"  parsed={'OK ' if parse(raw) else 'REJECT'} | {raw}")

zero-shot: format valid 45/120 (38%), correct among valid 40/45 (89%), correct overall 40/120 (33%)
few-shot: format valid 114/120 (95%), correct among valid 109/114 (96%), correct overall 109/120 (91%)

Five raw zero-shot replies (seed 10) - what would the parser do?
  parsed=REJECT | This looks like a Physics / Collision problem. I would rate the severity as High, and a good first step is to check collider and collision layer.
  parsed=OK  | Physics / Collision | High | Check collider and collision layer
  parsed=REJECT | This looks like a Performance problem. I would rate the severity as Medium, and a good first step is to profile the scene and check asset sizes.
  parsed=OK  | Performance | Medium | Profile the scene and check asset sizes
  parsed=OK  | Performance | Medium | Profile the scene and check asset sizes


In [18]:


def score_answer(reply):
    """0 = no cause, 1 = names the exception, 2 = names the exception and slot.item."""
    if "NullReferenceException" in reply and "slot.item" in reply:
        return 2
    if "NullReferenceException" in reply:
        return 1
    return 0


for items in [[], ["log"], ["log", "script"], ["log", "script", "version"]]:
    prompt = build_prompt(QUESTION, items)
    llm.set_seed(5)
    reply = ask(prompt)
    print(f"{items or 'none'}: {len(prompt)} chars, score {score_answer(reply)}")

none: 70 chars, score 0
['log']: 223 chars, score 1
['log', 'script']: 269 chars, score 2
['log', 'script', 'version']: 302 chars, score 2


In [19]:


def get_plan(goal, max_retries=3):
    """Return (plan, attempts_used). plan is None if every attempt failed."""
    prompt = plan_prompt(goal)
    for attempt in range(max_retries + 1):
        plan, problem = check_plan(ask(prompt))
        if plan is not None:
            return plan, attempt + 1
        # tell the model what was wrong (the reason from check_plan) and ask again
        prompt = (plan_prompt(goal) +
                  f"\nYour last answer was rejected: {problem}. "
                  "Fix it and reply with a JSON array only.")
    return None, max_retries + 1


GOALS = ["Create a simple enemy AI prototype", "Prepare a playable demo for Friday",
         "Add a save system to the game"]
first_try_ok = retries_used = failures = 0
llm.LOG.clear()
for seed in range(20, 45):
    llm.set_seed(seed)
    for goal in GOALS:
        plan, attempts = get_plan(goal)
        if plan is None:
            failures += 1
        elif attempts == 1:
            first_try_ok += 1
        retries_used += attempts - 1
total = 25 * len(GOALS)
task3_calls = len(llm.LOG)
print("plans asked for:", total)
print(f"valid on the first try: {first_try_ok} ({first_try_ok / total:.0%})")
print("retries used:", retries_used)
print("gave up after retries:", failures)
print("model calls in total:", task3_calls)
print(f"average calls per plan: {task3_calls / total:.2f}")

plans asked for: 75
valid on the first try: 53 (71%)
retries used: 22
gave up after retries: 0
model calls in total: 97
average calls per plan: 1.29


In [20]:
FILES = {"InventoryUI.cs": "40: foreach (var slot in slots)\n41: {\n42: label.text = "
                           "slot.item.name;\n43: }"}
TOOLS = {                      # the whitelist: the agent can run ONLY these
    "read_log": lambda arg: llm.LOG_TEXT,
    "search_file": lambda arg: FILES.get(arg, "file not found: " + arg),
    "check_version": lambda arg: "Unity 2022.3 LTS",
}
HEADER = ("You are a debugging agent. Available tools: read_log[], search_file[name], check_version[].\n"
          "Reply with 'Thought: ...' and then either 'Action: tool[argument]' or 'Final: answer'.\n")


def parse_step(reply):
    """Return ('final', text) or ('action', tool, arg) or ('bad', reply)."""
    final = re.search(r"Final:\s*(.+)", reply, re.S)
    if final:
        return ("final", final.group(1).strip())
    act = re.search(r"Action:\s*(\w+)\[(.*?)\]", reply)
    if act:
        return ("action", act.group(1), act.group(2))
    return ("bad", reply)


def short(text, n=60):
    """First line of text, cut to n characters, for the printed trace."""
    lines = text.splitlines()
    return (lines[0] if lines else "")[:n]


def react(question, max_steps=6):
    transcript = HEADER + "Question: " + question + "\n"       # AGENT: builds the prompt
    trace = []
    for step in range(1, max_steps + 1):                       # AGENT: the step limit
        reply = ask(transcript)                                # MODEL: the only model call
        kind = parse_step(reply)                               # AGENT: parse and validate
        if kind[0] == "final":                                 # case 1: Final answer
            trace.append(f"step {step}: Final -> {kind[1]}")
            return kind[1], trace
        if kind[0] == "bad":                                   # case 2: wrong format
            observation = "Bad format. Reply with Thought and then Action: tool[arg] or Final: answer."
            label = "bad reply"
        elif kind[1] not in TOOLS:                             # case 3: not whitelisted -> refuse
            observation = f"unknown tool {kind[1]}. Allowed: {', '.join(TOOLS)}"
            label = f"{kind[1]}[{kind[2]}]"
        else:                                                  # case 4: safe tool -> run it
            observation = TOOLS[kind[1]](kind[2])
            label = f"{kind[1]}[{kind[2]}]"
        trace.append(f"step {step}: {label} -> {short(observation)}")
        transcript += reply + "\nObservation: " + observation + "\n"   # AGENT: records what happened
    return None, trace + ["stopped: step limit reached"]


QUESTION4 = "Why does the game crash when the inventory opens?"
DEMO_SEED = 15
llm.set_seed(DEMO_SEED)
answer, trace = react(QUESTION4)
print("\n".join(trace))
print("answer:", answer)

outcomes = {"solved": 0, "step limit": 0}
unknown_tool_seen = 0
react_calls = []
for seed in range(100, 130):
    llm.set_seed(seed)
    llm.LOG.clear()
    answer, trace = react(QUESTION4)
    react_calls.append(len(llm.LOG))
    outcomes["solved" if answer else "step limit"] += 1
    unknown_tool_seen += any("unknown tool" in t for t in trace)
print("30 normal runs:", outcomes, "| runs that tried an unknown tool:", unknown_tool_seen)
print(f"model calls per solved run: mean {statistics.mean(react_calls):.2f}, "
      f"min {min(react_calls)}, max {max(react_calls)}")

llm.set_seed(1, mode="stuck")
llm.LOG.clear()
answer, trace = react(QUESTION4)
print("stuck model:", answer, "|", trace[-1], "| steps:", len(trace) - 1)
stuck_calls = len(llm.LOG)
llm.set_seed(0)                      # back to normal mode for later tasks

step 1: read_log[] -> ERROR NullReferenceException: Object reference not set to an
step 2: run_game[] -> unknown tool run_game. Allowed: read_log, search_file, check
step 3: search_file[InventoryUI.cs] -> 40: foreach (var slot in slots)
step 4: Final -> In InventoryUI.cs line 42, check that slot.item is not null before reading name.
answer: In InventoryUI.cs line 42, check that slot.item is not null before reading name.
30 normal runs: {'solved': 30, 'step limit': 0} | runs that tried an unknown tool: 7
model calls per solved run: mean 3.37, min 3, max 6
stuck model: None | stopped: step limit reached | steps: 6


In [21]:
TRUSTED = {"Physics.Raycast", "transform.Translate", "Destroy", "audioSource.Play", "transform.Find"}
TASKS = ["cast a ray forward from the player", "move the character with its speed",
         "destroy the object after two seconds", "play the sound attached to this object",
         "find the child object called Weapon"]


def called_names(line):
    return re.findall(r"([A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*)\s*\(", line)


def unsupported(line):
    """Return the called names in the line that are not in TRUSTED."""
    return [n for n in called_names(line) if n not in TRUSTED]


llm.set_seed(10)
for task in TASKS:
    line = ask(f"Write one line of C# to {task}.")
    bad = unsupported(line)
    print(f"UNSUPPORTED {bad}" if bad else "supported", "|", line)

checked = flagged = wrongly_used = 0
for seed in range(50, 100):
    llm.set_seed(seed)
    for task in TASKS:
        line = ask(f"Write one line of C# to {task}.")
        checked += 1
        flagged += bool(unsupported(line))
        wrongly_used += line == "Destroy(2f, gameObject);"
print(f"lines checked: {checked}, flagged as unsupported: {flagged} ({flagged / checked:.1%})")
print("real function used wrongly and NOT flagged:", wrongly_used)

supported | Physics.Raycast(origin, direction, out hit);
supported | transform.Translate(move * speed * Time.deltaTime);
supported | Destroy(gameObject, 2f);
UNSUPPORTED ['audioSource.PlayLoudly'] | audioSource.PlayLoudly();
supported | transform.Find("Weapon");
lines checked: 250, flagged as unsupported: 66 (26.4%)
real function used wrongly and NOT flagged: 2


In [22]:
zv, zc, zt = task1["zero-shot"]
fv, fc, ft = task1["few-shot"]
print(f"zero-shot : 1 call per answer | {zt - zv}/{zt} replies had an invalid format "
      f"({(zt - zv) / zt:.0%}); {zv - zc}/{zv} valid replies had the wrong category")
print(f"few-shot  : 1 call per answer | {ft - fv}/{ft} replies had an invalid format "
      f"({(ft - fv) / ft:.0%}); {fv - fc}/{fv} valid replies had the wrong category")
print(f"plan+retry: {task3_calls / total:.2f} calls per plan ({task3_calls} calls / {total} plans) | "
      f"{total - first_try_ok}/{total} first answers unusable; {retries_used} retries; {failures} gave up")
print(f"ReAct     : {statistics.mean(react_calls):.2f} calls per answer (30 runs, range "
      f"{min(react_calls)}-{max(react_calls)}) | unknown tool in {unknown_tool_seen}/30 runs; "
      f"stuck model used {stuck_calls} calls and stopped at the step limit")
print("\nNote: all numbers above come from the offline mock backend (not a language model).")

zero-shot : 1 call per answer | 75/120 replies had an invalid format (62%); 5/45 valid replies had the wrong category
few-shot  : 1 call per answer | 6/120 replies had an invalid format (5%); 5/114 valid replies had the wrong category
plan+retry: 1.29 calls per plan (97 calls / 75 plans) | 22/75 first answers unusable; 22 retries; 0 gave up
ReAct     : 3.37 calls per answer (30 runs, range 3-6) | unknown tool in 7/30 runs; stuck model used 6 calls and stopped at the step limit

Note: all numbers above come from the offline mock backend (not a language model).
